# PeCab 명사 코퍼스 생성

- 목적: DART 사업보고서 XML에서 주요 섹션을 추출한 뒤 PeCab 한국어 명사 코퍼스 생성
- 입력: `data/dart/raw_xml/*.xml`
- 출력: `data/noun_corpus_pecab.csv`, `data/noun_corpus_pecab.pkl`
- 병렬: `ProcessPoolExecutor(max_workers=6)` 강제 사용

In [4]:
from pathlib import Path
import html
import re
import sys
import subprocess
import warnings
from concurrent.futures import ProcessPoolExecutor, as_completed

import pandas as pd
from tqdm.auto import tqdm

try:
    from pecab import PeCab
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pecab"])
    from pecab import PeCab

warnings.filterwarnings("ignore", message="overflow encountered in scalar add")

if Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive")

LOCAL_BASE_DIR = Path(".")
BASE_CANDIDATES = [
    Path("/content/drive/MyDrive/UD_26"),
    Path("/content/drive/My Drive/UD_26"),
    LOCAL_BASE_DIR,
    LOCAL_BASE_DIR.parent,
]


def first_existing(candidates, default=None):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return default if default is not None else candidates[0]


BASE_DIR = first_existing(
    [p for p in BASE_CANDIDATES if (p / "data").exists() or (p / "final").exists()],
    LOCAL_BASE_DIR,
)

DRIVE_FINAL_DIR = BASE_DIR / "final"
OUTPUT_DIR = BASE_DIR / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_XML_DIR = first_existing([
    DRIVE_FINAL_DIR / "raw_xml",
    BASE_DIR / "data/dart/raw_xml",
    BASE_DIR / "raw_xml",
])

MAX_WORKERS = 6

xml_files = sorted(RAW_XML_DIR.glob("*.xml"))
print("BASE_DIR:", BASE_DIR)
print("RAW_XML_DIR:", RAW_XML_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("XML files:", len(xml_files))
print("MAX_WORKERS:", MAX_WORKERS)

if not xml_files:
    raise FileNotFoundError(f"No XML files found in {RAW_XML_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE_DIR: /content/drive/MyDrive/UD_26
RAW_XML_DIR: /content/drive/MyDrive/UD_26/final/raw_xml
OUTPUT_DIR: /content/drive/MyDrive/UD_26/data
XML files: 381
MAX_WORKERS: 6


In [5]:
def extract_target_sections(xml_text: str) -> tuple[str, int]:
    title_re = re.compile(r"<TITLE\b[^>]*>(.*?)</TITLE>", flags=re.I | re.S)
    main_title_re = re.compile(
        r"^\s*(I|II|III|IV|V|VI|VII|VIII|IX|X|Ⅰ|Ⅱ|Ⅲ|Ⅳ|Ⅴ|Ⅵ|Ⅶ|Ⅷ|Ⅸ|Ⅹ)\."
    )
    target_title_regex = {
        "II. 사업의 내용": r"^(II|Ⅱ)\.\s*사업의\s*내용",
        "IV. 이사의 경영진단 및 분석의견": r"^(IV|Ⅳ)\.\s*이사의\s*경영진단\s*및\s*분석의견",
        "VI. 이사회 등 회사의 기관에 관한 사항": r"^(VI|Ⅵ)\.\s*이사회\s*등\s*회사의\s*기관에\s*관한\s*사항",
    }

    titles = []
    for match in title_re.finditer(xml_text):
        title = html.unescape(re.sub(r"\s+", " ", match.group(1))).strip()
        if main_title_re.match(title):
            titles.append((title, match.start()))

    sections = []
    seen_sections = set()

    for i, (title, start) in enumerate(titles):
        section_name = None
        for name, pattern in target_title_regex.items():
            if re.search(pattern, title):
                section_name = name
                break

        if section_name is None:
            continue

        end = titles[i + 1][1] if i + 1 < len(titles) else len(xml_text)
        section_text = re.sub(r"<[^>]+>", " ", xml_text[start:end])
        section_text = html.unescape(re.sub(r"\s+", " ", section_text)).strip()
        sections.append(section_text)
        seen_sections.add(section_name)

    return " ".join(sections), len(seen_sections)


def parse_xml_metadata(path: Path) -> tuple[str, int, str]:
    match = re.match(r"(\d{6})_(\d{4})_(\d+)\.xml$", path.name)
    if not match:
        raise ValueError(f"Unexpected XML file name: {path.name}")
    stock_code, fiscal_year, rcept_no = match.groups()
    return stock_code, int(fiscal_year), rcept_no

In [6]:
def process_xml_path(path_text: str) -> dict:
    from pathlib import Path
    from pecab import PeCab

    path = Path(str(path_text).replace("\\", "/"))
    stock_code, fiscal_year, rcept_no = parse_xml_metadata(path)
    xml_text = path.read_text(encoding="utf-8", errors="ignore")
    document, section_count = extract_target_sections(xml_text)

    tagger = PeCab()
    noun_tokens = [noun for noun in tagger.nouns(document) if len(noun) > 1]

    return {
        "stock_code": stock_code,
        "fiscal_year": fiscal_year,
        "rcept_no": rcept_no,
        "file_name": path.name,
        "section_count": section_count,
        "document": document,
        "total_word_count": len(document.split()),
        "noun_tokens": noun_tokens,
        "noun_document": " ".join(noun_tokens),
        "noun_token_count": len(noun_tokens),
    }

In [ ]:
path_texts = [str(path) for path in xml_files]
rows = []

print(f"Starting PeCab noun extraction with {MAX_WORKERS} workers")

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_xml_path, path_text): path_text for path_text in path_texts}

    for future in tqdm(as_completed(futures), total=len(futures), desc="PeCab files"):
        path_text = futures[future]
        try:
            rows.append(future.result())
        except Exception as exc:
            print("Failed:", path_text, repr(exc))
            raise

noun_corpus_df = pd.DataFrame(rows)
noun_corpus_df = noun_corpus_df.sort_values(["stock_code", "fiscal_year", "rcept_no"]).reset_index(drop=True)

print("Rows:", len(noun_corpus_df))
display(noun_corpus_df[["stock_code", "fiscal_year", "rcept_no", "section_count", "total_word_count", "noun_token_count"]].head())
display(noun_corpus_df["section_count"].value_counts().sort_index().rename_axis("section_count").reset_index(name="row_count"))

Starting PeCab noun extraction with 6 workers


PeCab files:   0%|          | 0/381 [00:00<?, ?it/s]

In [ ]:
csv_path = OUTPUT_DIR / "noun_corpus_pecab.csv"
pkl_path = OUTPUT_DIR / "noun_corpus_pecab.pkl"

noun_corpus_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
noun_corpus_df.to_pickle(pkl_path)

print("Saved CSV:", csv_path.resolve())
print("Saved PKL:", pkl_path.resolve())